# 23_26년 서비스오더.xlsx → pkl 변환 및 정합성 검증
DocuONE 경로에서 xlsx를 읽어 `c:\DA\서비스오더.pkl`로 저장하고,
신규아파트 오더 수와 전체 오더 수의 정합성을 검증합니다.

In [ ]:
import os
import pandas as pd

XLSX_PATH  = r'\\DocuONE\MyDrive\개인함\.xlsx'
OUTPUT_PKL = r'c:\DA\서비스오더.pkl'

print(f'파일 경로: {XLSX_PATH}')
print(f'파일 존재: {os.path.exists(XLSX_PATH)}')
if os.path.exists(XLSX_PATH):
    print(f'파일 크기: {os.path.getsize(XLSX_PATH)/1024/1024:.1f} MB')

파일 경로: \\DocuONE\MyDrive\개인함\23_26년_서비스오더\23_26년_서비스오더.xlsx
파일 존재: True
파일 크기: 417.1 MB


In [ ]:
# ── 시트 목록 확인 ──
xl = pd.ExcelFile(XLSX_PATH, engine='openpyxl')
sheet_names = xl.sheet_names
print(f'총 {len(sheet_names)}개 시트:')
for i, s in enumerate(sheet_names):
    print(f'  [{i:02d}] {s}')

In [ ]:
# ── 원시 오더 시트만 선별 (월별 데이터 시트) ──
# '서비스오더(YY.MM월)' 패턴 + '_RawData' 시트만 포함
import re

raw_sheets = [s for s in sheet_names
              if re.search(r'서비스오더\s*\(\d{2}', s) or s == '_RawData']
other_sheets = [s for s in sheet_names if s not in raw_sheets]

print(f'원시 오더 시트 ({len(raw_sheets)}개):')
for s in raw_sheets: print(f'  {s}')
print(f'\n집계/기타 시트 ({len(other_sheets)}개):')
for s in other_sheets: print(f'  {s}')

In [ ]:
# ── 원시 오더 시트 읽기 & 병합 ──
# raw_sheets가 비어있으면 전체 시트 사용
target_sheets = raw_sheets if raw_sheets else sheet_names

dfs = []
for sheet in target_sheets:
    print(f'읽는 중: {sheet} ...', end=' ', flush=True)
    try:
        df_s = pd.read_excel(XLSX_PATH, sheet_name=sheet, engine='openpyxl', dtype=str)
        df_s['시트명'] = sheet
        dfs.append(df_s)
        print(f'{len(df_s):,}행')
    except Exception as e:
        print(f'오류: {e}')

print(f'\n병합 중...')
df_all = pd.concat(dfs, ignore_index=True)
print(f'전체 행수: {len(df_all):,}')
print(f'컬럼: {list(df_all.columns)}')
df_all.head(3)

In [ ]:
# ── 날짜/숫자 타입 변환 ──
DATE_COLS = ['고객방문일','오더생성일','마감일','전입/전출일','접수일자']
for col in DATE_COLS:
    if col in df_all.columns:
        df_all[col] = pd.to_datetime(df_all[col], errors='coerce')
        print(f'날짜 변환: {col}')

if '수수료금액' in df_all.columns:
    df_all['수수료금액'] = pd.to_numeric(df_all['수수료금액'], errors='coerce')

print(f'\n최종 shape: {df_all.shape}')

In [ ]:
# ── pkl 저장 ──
df_all.to_pickle(OUTPUT_PKL)
sz = os.path.getsize(OUTPUT_PKL)/1024/1024
print(f'저장 완료: {OUTPUT_PKL} ({sz:.0f} MB, {len(df_all):,}행)')

---
## 정합성 검증
신규아파트 리스트 오더 수 vs 전체 오더 수 비교

In [8]:
# ── 신규아파트 리스트 로드 ──
import re as _re

NEW_APT_TXT = r'c:\DA\23_26_신규아파트리스트.txt'
APT_TXT     = r'c:\DA\확정_아파트_리스트.txt'

def clean_addr(s):
    return ' '.join(_re.sub(r'\(.*?\)', '', str(s)).split()).strip()

def load_list(path):
    addrs = set()
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line: addrs.add(clean_addr(line))
    return addrs

new_apt_set    = load_list(NEW_APT_TXT)
normal_apt_set = load_list(APT_TXT)

print(f'신규아파트 리스트: {len(new_apt_set)}개 주소')
print(f'확정아파트 리스트: {len(normal_apt_set)}개 주소')

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\DA\\23_26_신규아파트리스트.txt'

In [ ]:
# ── pkl 재로드 ──
df = pd.read_pickle(OUTPUT_PKL)
df['clean_addr'] = df['주소'].apply(lambda x: clean_addr(x) if pd.notnull(x) else '')

# 연도 추출
date_col = next((c for c in DATE_COLS if c in df.columns), None)
if date_col:
    df['year'] = pd.to_datetime(df[date_col], errors='coerce').dt.year

print(f'총 오더: {len(df):,}건')
if 'year' in df.columns:
    print('\n연도별 분포:')
    print(df['year'].value_counts().sort_index().to_dict())

In [ ]:
# ── 유형 분류 ──
def classify(addr_key):
    if addr_key in new_apt_set:    return '신규아파트'
    if addr_key in normal_apt_set: return '아파트'
    if any(kw in addr_key for kw in ['아파트','래미안','자이','힐스테이트','푸르지오']):
        return '아파트(키워드)'
    return '비아파트'

df['유형'] = df['clean_addr'].apply(classify)

print('== 유형별 오더 수 ==')
type_cnt = df['유형'].value_counts()
for t, c in type_cnt.items():
    pct = c / len(df) * 100
    print(f'  {t:<15} {c:>10,}건 ({pct:>5.1f}%)')
print(f'  {"합계":<15} {len(df):>10,}건')

In [ ]:
# ── 신규아파트 상세 검증 ──
df_new = df[df['유형'] == '신규아파트'].copy()
print(f'신규아파트 오더: {len(df_new):,}건')

# 주소별 건수
addr_cnt = df_new['clean_addr'].value_counts()
print(f'\n신규아파트 주소 수: {len(addr_cnt)}개 (리스트: {len(new_apt_set)}개)')

# 리스트에 있지만 오더가 없는 주소
no_order = new_apt_set - set(addr_cnt.index)
print(f'오더 없는 신규아파트: {len(no_order)}개')
if no_order:
    for a in list(no_order)[:10]: print(f'  - {a}')

print(f'\n연도별 신규아파트 오더:')
if 'year' in df_new.columns:
    print(df_new['year'].value_counts().sort_index().to_dict())

print(f'\n오더 TOP 10 신규아파트:')
print(addr_cnt.head(10).to_frame('오더수').to_string())

In [ ]:
# ── 센터별 / 소분류별 교차 집계 ──
center_col = '고객서비스처리센터' if '고객서비스처리센터' in df.columns else '서비스처리센터'

CENTER_NAMES = {
    'H051':'대전북부센터', 'H061':'특수고객센터',
    'H071':'대전중부서센터', 'H072':'대전중부동센터',
    'H073':'대전서부센터',  'H074':'대전동부센터',
    'H075':'대전남부센터',  'H076':'대전남부2센터',
}

print('== 센터별 × 유형별 오더 교차표 ==')
ct = pd.crosstab(df[center_col], df['유형'], margins=True, margins_name='합계')
ct.index = [f'{CENTER_NAMES.get(i,i)} ({i})' if i!='합계' else i for i in ct.index]
print(ct.to_string())

print('\n== 소분류별 전체 오더 수 ==')
if '중분류' in df.columns:
    print(df['중분류'].value_counts().head(15).to_string())

In [ ]:
# ── 최종 요약 ──
print('='*50)
print('정합성 검증 최종 요약')
print('='*50)
print(f'xlsx 파일    : {XLSX_PATH}')
print(f'pkl 저장 경로: {OUTPUT_PKL}')
print(f'전체 오더 수  : {len(df):,}건')
print()
print(f'신규아파트 리스트 주소: {len(new_apt_set)}개')
print(f'신규아파트 오더 건수  : {len(df_new):,}건')
print(f'신규아파트 비율       : {len(df_new)/len(df)*100:.2f}%')
print()
print(f'아파트 오더 합계      : {type_cnt.get("아파트",0)+type_cnt.get("아파트(키워드)",0):,}건')
print(f'비아파트 오더 건수    : {type_cnt.get("비아파트",0):,}건')
if 'year' in df.columns:
    print()
    print('연도별 분포:')
    for yr, cnt in df['year'].value_counts().sort_index().items():
        print(f'  {yr}년: {cnt:,}건')
print('='*50)